# LoRA vs No-LoRA — Comparison using Saved Results

## Summary of available data

| Source | Type | Period | Coverage |
|---|---|---|---|
| **TensorBoard (pre-LoRA)** | Within-DB validation AUC (best epoch) | Apr 30 – May 7, 2026 | ViT: 5/5 DBs · EfficientNet: 4/5 · CNN-SRM: 3/5 |
| **pkl cross_db_eval (LoRA)** | Full 5×5 cross-database evaluation | May 25 – Jun 9, 2026 | All models: 5/5 DBs |

> **Note:** No pre-LoRA cross-database evaluation results exist on the server — the 5×5 cross-DB evaluation was first introduced together with LoRA.  
> The within-DB comparison is therefore the only **direct apples-to-apples** comparison available.


In [1]:
# =================== CELL 1: SETUP ===================
import os, pickle, warnings, subprocess
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

DATABASES = ['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'Combined']
TB        = '/home/sceuser/RealEyes/tensorboard_logs'
PKL_ROOT  = '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/RealEyes_experiment'
COMPARE_DIR = '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora'
os.makedirs(COMPARE_DIR, exist_ok=True)

# ── helpers ────────────────────────────────────────────────────────────────
def read_val_metrics(logdir, max_events=300):
    """Return best validation AUC / Accuracy from a TensorBoard run folder."""
    metrics = {}
    val_dir = os.path.join(logdir, 'validation')
    search  = val_dir if os.path.exists(val_dir) else logdir
    for root, _, files in os.walk(search):
        for f in sorted(files):
            if not f.startswith('events'):
                continue
            try:
                ds = tf.data.TFRecordDataset(os.path.join(root, f), compression_type='')
                for raw in ds.take(max_events):
                    e = tf.compat.v1.Event()
                    e.ParseFromString(raw.numpy())
                    for v in e.summary.value:
                        if v.HasField('simple_value'):
                            metrics.setdefault(v.tag, []).append(float(v.simple_value))
                        elif v.HasField('tensor'):
                            try:
                                metrics.setdefault(v.tag, []).append(
                                    float(tf.make_ndarray(v.tensor)))
                            except: pass
            except: pass
    return metrics

def best_val(metrics_dict, metric='auc'):
    vals = metrics_dict.get(f'epoch_{metric}',
           metrics_dict.get(f'evaluation_{metric}_vs_iterations', []))
    return max(vals) if vals else None

# ── Pre-LoRA run paths (last / best run before LoRA was added) ─────────────
PRE_LORA_RUNS = {
    'ViT': {
        'OpenForensics': f'{TB}/vit/train_OpenForensics/20260504_0743',
        'CustomWar':     f'{TB}/vit/train_CustomWar/20260504_0850',
        'CelebDF':       f'{TB}/vit/train_CelebDF/20260504_0928',
        'CiFake':        f'{TB}/vit/train_CiFake/20260504_1127',
        'Combined':      f'{TB}/vit/train_Combined/20260504_1311',
    },
    'EfficientNet': {
        'OpenForensics': f'{TB}/efficientnet/train_OpenForensics/20260505_1104',
        'CustomWar':     f'{TB}/efficientnet/train_CustomWar/20260505_1411',
        'CelebDF':       f'{TB}/efficientnet/train_CelebDF/20260505_1443',
        'CiFake':        None,
        'Combined':      f'{TB}/efficientnet/train_ALL/20260505_1537',
    },
    'CNN-SRM': {
        'OpenForensics': f'{TB}/cnn_srm/train_OpenForensics/20260506_1630',
        'CustomWar':     f'{TB}/cnn_srm/train_CustomWar/20260507_0604',
        'CelebDF':       f'{TB}/cnn_srm/train_CelebDF/20260507_1742',
        'CiFake':        None,
        'Combined':      None,
    },
}

# ── Load pre-LoRA data from TensorBoard ────────────────────────────────────
pre_lora = {}
print("Loading pre-LoRA TensorBoard data...")
for model, db_runs in PRE_LORA_RUNS.items():
    pre_lora[model] = {}
    for db, path in db_runs.items():
        if path is None or not os.path.exists(path):
            pre_lora[model][db] = None
            continue
        m = read_val_metrics(path)
        auc = best_val(m, 'auc')
        acc = best_val(m, 'accuracy')
        pre_lora[model][db] = {'roc_auc': auc, 'accuracy': acc}
        print(f"  {model}/{db}: AUC={auc:.4f}  Acc={acc:.4f}" if auc else f"  {model}/{db}: no data")

# ── Load LoRA data from pkl ─────────────────────────────────────────────────
lora_pkl = {}
print("\nLoading LoRA pkl results...")
for model, name in [('CNN-SRM','cnn_srm'), ('EfficientNet','efficientnet'), ('ViT','vit')]:
    lora_pkl[model] = pickle.load(open(f'{PKL_ROOT}/{name}/all_results.pkl','rb'))
    print(f"  {model}: {len(lora_pkl[model])} training databases × evaluations loaded")

print("\nSetup complete.")


2026-06-09 12:12:19.876178: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading pre-LoRA TensorBoard data...


I0000 00:00:1781007142.045804  416595 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13659 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-06-09 12:12:22.399974: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144
2026-06-09 12:12:22.419634: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-06-09 12:12:22.489620: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


  ViT/OpenForensics: AUC=0.5293  Acc=0.5019
  ViT/CustomWar: AUC=0.9297  Acc=0.8596
  ViT/CelebDF: AUC=0.7008  Acc=0.8717
  ViT/CiFake: AUC=0.6650  Acc=0.5002


2026-06-09 12:12:22.619062: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


  ViT/Combined: AUC=0.6510  Acc=0.4627
  EfficientNet/OpenForensics: AUC=0.9572  Acc=0.7407


2026-06-09 12:12:22.885747: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


  EfficientNet/CustomWar: AUC=0.9999  Acc=0.9934
  EfficientNet/CelebDF: AUC=0.7663  Acc=0.4550
  EfficientNet/Combined: AUC=0.8579  Acc=0.5594


  CNN-SRM/OpenForensics: AUC=0.8707  Acc=0.5690
  CNN-SRM/CustomWar: AUC=0.9985  Acc=0.9868
  CNN-SRM/CelebDF: AUC=0.6858  Acc=0.1339

Loading LoRA pkl results...
  CNN-SRM: 5 training databases × evaluations loaded
  EfficientNet: 5 training databases × evaluations loaded
  ViT: 5 training databases × evaluations loaded

Setup complete.


2026-06-09 12:12:23.284033: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [2]:
# =================== CELL 2: WITHIN-DB AUC COMPARISON ===================
# Compares best validation AUC (pre-LoRA) vs test AUC (LoRA) on the SAME database

MODELS = ['CNN-SRM', 'EfficientNet', 'ViT']
colors = {'Pre-LoRA (no adapters)': '#2196F3', 'LoRA (current)': '#F44336'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Within-Database AUC: Pre-LoRA vs LoRA\n'
             '(Pre-LoRA = best validation epoch · LoRA = test-time evaluation)',
             fontsize=13, fontweight='bold', y=1.02)

all_rows = []

for ax, model in zip(axes, MODELS):
    nl_vals, lr_vals, labels = [], [], []
    
    for db in DATABASES:
        nl = pre_lora[model].get(db)
        lr_row = lora_pkl[model].get(db, {}).get(db, {})
        lr = lr_row.get('roc_auc', None) if lr_row else None
        
        nl_v = nl['roc_auc'] if nl else np.nan
        lr_v = lr if lr is not None else np.nan
        
        nl_vals.append(nl_v)
        lr_vals.append(lr_v)
        labels.append(db)
        
        if not np.isnan(nl_v) and not np.isnan(lr_v):
            all_rows.append({'Model': model, 'Database': db,
                             'Pre-LoRA AUC': nl_v, 'LoRA AUC': lr_v,
                             'Δ AUC': lr_v - nl_v})
    
    x = np.arange(len(DATABASES))
    w = 0.35
    
    b1 = ax.bar(x - w/2, nl_vals, w, label='Pre-LoRA', color='#2196F3', alpha=0.85, zorder=3)
    b2 = ax.bar(x + w/2, lr_vals, w, label='LoRA',     color='#F44336', alpha=0.85, zorder=3)
    
    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h) and h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=6.5, fontweight='bold')
    
    ax.set_title(model, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('ROC-AUC'); ax.set_ylim(0, 1.12)
    ax.legend(fontsize=8)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout()
out = os.path.join(COMPARE_DIR, 'within_db_auc_comparison.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {out}")

# Print summary
print("\n=== Within-DB AUC Delta (LoRA - Pre-LoRA) ===")
df = pd.DataFrame(all_rows)
if not df.empty:
    print(df[['Model','Database','Pre-LoRA AUC','LoRA AUC','Δ AUC']].to_string(index=False))
    df.to_csv(os.path.join(COMPARE_DIR, 'within_db_comparison.csv'), index=False)


Saved: /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora/within_db_auc_comparison.png

=== Within-DB AUC Delta (LoRA - Pre-LoRA) ===
       Model      Database  Pre-LoRA AUC  LoRA AUC     Δ AUC
     CNN-SRM OpenForensics      0.870703  0.522270 -0.348433
     CNN-SRM     CustomWar      0.998544  0.972848 -0.025696
     CNN-SRM       CelebDF      0.685846  0.656624 -0.029222
EfficientNet OpenForensics      0.957169  0.803074 -0.154094
EfficientNet     CustomWar      0.999879  0.999850 -0.000029
EfficientNet       CelebDF      0.766254  0.579349 -0.186905
         ViT OpenForensics      0.529305  0.605971  0.076666
         ViT     CustomWar      0.929660  0.973159  0.043499
         ViT       CelebDF      0.700771  0.629737 -0.071034
         ViT        CiFake      0.664983  0.716130  0.051147
         ViT      Combined      0.651043  0.787610  0.136567


In [3]:
# =================== CELL 3: DELTA CHART (LoRA − Pre-LoRA) ===================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('LoRA Impact on Within-DB AUC  (positive = LoRA improved, negative = LoRA hurt)',
             fontsize=13, fontweight='bold')

for ax, model in zip(axes, MODELS):
    deltas, labels, colors_list = [], [], []
    
    for db in DATABASES:
        nl = pre_lora[model].get(db)
        lr_row = lora_pkl[model].get(db, {}).get(db, {})
        lr = lr_row.get('roc_auc', None) if lr_row else None
        
        if nl is None or lr is None:
            continue
        delta = lr - nl['roc_auc']
        deltas.append(delta)
        labels.append(db)
        colors_list.append('#27AE60' if delta > 0.01 else '#E74C3C' if delta < -0.01 else '#95A5A6')
    
    if not deltas:
        ax.text(0.5, 0.5, 'No comparable data', ha='center', transform=ax.transAxes)
        continue
    
    bars = ax.barh(labels, deltas, color=colors_list, alpha=0.85, height=0.55, edgecolor='white', linewidth=0.5)
    
    for bar, val in zip(bars, deltas):
        ax.text(val + (0.003 if val >= 0 else -0.003),
                bar.get_y() + bar.get_height()/2,
                f'{val:+.3f}', va='center',
                ha='left' if val >= 0 else 'right', fontsize=9, fontweight='bold')
    
    ax.axvline(0, color='black', linewidth=1.2)
    ax.set_title(model, fontsize=12, fontweight='bold')
    ax.set_xlabel('ΔAUC (LoRA − Pre-LoRA)')
    ax.set_xlim(-0.5, 0.5)
    ax.grid(axis='x', alpha=0.3)

    improved = sum(1 for d in deltas if d > 0.01)
    worsened = sum(1 for d in deltas if d < -0.01)
    neutral  = len(deltas) - improved - worsened
    ax.set_xlabel(f'ΔAUC   ▲{improved} improved  ▼{worsened} worsened  ={neutral} neutral')

plt.tight_layout()
out = os.path.join(COMPARE_DIR, 'within_db_delta.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {out}")


Saved: /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora/within_db_delta.png


In [4]:
# =================== CELL 4: LoRA CROSS-DB HEATMAPS ===================
# Pre-LoRA cross-db data does not exist — only LoRA results shown
# Row = Train DB, Col = Test DB, diagonal = within-DB

for metric, label in [('roc_auc','ROC-AUC'), ('accuracy','Accuracy'), ('f1_fake','F1-FAKE')]:
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    fig.suptitle(f'LoRA Cross-Database Evaluation — {label}\n'
                 f'(no pre-LoRA cross-DB baseline exists)', fontsize=13, fontweight='bold')
    
    for ax, model in zip(axes, MODELS):
        mat = np.full((5, 5), np.nan)
        for i, train_db in enumerate(DATABASES):
            for j, test_db in enumerate(DATABASES):
                r = lora_pkl[model].get(train_db, {}).get(test_db)
                if r:
                    mat[i, j] = r.get(metric, np.nan)
        
        df_mat = pd.DataFrame(mat, index=[f'Train:{d}' for d in DATABASES],
                              columns=[f'Test:{d}' for d in DATABASES])
        
        # Bold diagonal
        mask_nan = np.isnan(mat)
        sns.heatmap(df_mat.astype(float), annot=True, fmt='.3f',
                    cmap='Blues', vmin=0.3, vmax=1.0, mask=mask_nan,
                    ax=ax, linewidths=0.5, cbar_kws={'label': label})
        ax.set_title(model); ax.set_xlabel('Test DB'); ax.set_ylabel('Train DB')
        ax.tick_params(axis='x', rotation=45); ax.tick_params(axis='y', rotation=0)
        
        # Highlight diagonal
        for k in range(5):
            ax.add_patch(plt.Rectangle((k, k), 1, 1, fill=False,
                                       edgecolor='gold', lw=2.5, zorder=5))
    
    plt.tight_layout()
    out = os.path.join(COMPARE_DIR, f'lora_cross_db_{metric}.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {out}")


Saved: /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora/lora_cross_db_roc_auc.png


Saved: /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora/lora_cross_db_accuracy.png


Saved: /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora/lora_cross_db_f1_fake.png


In [5]:
# =================== CELL 5: LoRA CROSS-DB GENERALIZATION ===================
# Average off-diagonal AUC per training DB — measures generalization to unseen DBs

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('LoRA Cross-DB Generalization (avg. off-diagonal AUC)\nModel trained on one DB, tested on all others',
             fontsize=13, fontweight='bold')

for ax, model in zip(axes, MODELS):
    within_auc, cross_auc, labels = [], [], []
    
    for i, train_db in enumerate(DATABASES):
        within_vals, cross_vals = [], []
        for j, test_db in enumerate(DATABASES):
            r = lora_pkl[model].get(train_db, {}).get(test_db)
            if not r:
                continue
            v = r.get('roc_auc', np.nan)
            if i == j:
                within_vals.append(v)
            else:
                cross_vals.append(v)
        if within_vals or cross_vals:
            within_auc.append(np.nanmean(within_vals) if within_vals else np.nan)
            cross_auc.append(np.nanmean(cross_vals) if cross_vals else np.nan)
            labels.append(train_db)
    
    x = np.arange(len(labels))
    w = 0.35
    b1 = ax.bar(x - w/2, within_auc, w, label='Within-DB',  color='#3498DB', alpha=0.85)
    b2 = ax.bar(x + w/2, cross_auc,  w, label='Cross-DB',   color='#E67E22', alpha=0.85)
    
    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h):
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=7)
    
    ax.set_title(model); ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('ROC-AUC'); ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8); ax.axhline(0.5, color='gray', linestyle='--', lw=0.8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
out = os.path.join(COMPARE_DIR, 'lora_generalization.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {out}")


Saved: /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora/lora_generalization.png


In [6]:
# =================== CELL 6: SUMMARY TABLE ===================

rows = []
for model in MODELS:
    # Pre-LoRA avg within-DB
    pre_aucs = [pre_lora[model][db]['roc_auc']
                for db in DATABASES
                if pre_lora[model].get(db) and pre_lora[model][db]['roc_auc'] is not None]
    
    # LoRA avg within-DB (diagonal of cross_db_eval)
    lr_within_aucs = []
    lr_cross_aucs  = []
    for i, train_db in enumerate(DATABASES):
        for j, test_db in enumerate(DATABASES):
            r = lora_pkl[model].get(train_db, {}).get(test_db)
            if r:
                v = r.get('roc_auc', np.nan)
                if i == j:
                    lr_within_aucs.append(v)
                else:
                    lr_cross_aucs.append(v)
    
    pre_avg  = np.nanmean(pre_aucs)        if pre_aucs       else np.nan
    lr_w_avg = np.nanmean(lr_within_aucs)  if lr_within_aucs else np.nan
    lr_c_avg = np.nanmean(lr_cross_aucs)   if lr_cross_aucs  else np.nan
    delta_w  = lr_w_avg - pre_avg
    
    rows.append({
        'Model':                model,
        'Pre-LoRA Within-DB AUC': round(float(pre_avg), 4),
        'LoRA Within-DB AUC':     round(float(lr_w_avg), 4),
        'Δ Within-DB':            round(float(delta_w), 4),
        'LoRA Cross-DB AUC':      round(float(lr_c_avg), 4),
        'Pre-LoRA DBs available': len(pre_aucs),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Visual table
fig, ax = plt.subplots(figsize=(16, 2.5))
ax.axis('off')
tbl = ax.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.3, 2.0)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2C3E50'); cell.set_text_props(color='white', fontweight='bold')
    elif c == 3:  # Δ column
        try:
            v = float(df.iloc[r-1, c])
            cell.set_facecolor('#D5F5E3' if v > 0.005 else '#FADBD8' if v < -0.005 else '#FDFEFE')
        except: pass
plt.tight_layout()
plt.savefig(os.path.join(COMPARE_DIR, 'summary_table.png'), dpi=150, bbox_inches='tight')
plt.close()
print(f"\nSaved summary table")
df.to_csv(os.path.join(COMPARE_DIR, 'summary.csv'), index=False)


       Model  Pre-LoRA Within-DB AUC  LoRA Within-DB AUC  Δ Within-DB  LoRA Cross-DB AUC  Pre-LoRA DBs available
     CNN-SRM                  0.8517              0.7305      -0.1212             0.4931                       3
EfficientNet                  0.8953              0.8293      -0.0660             0.5225                       4
         ViT                  0.6952              0.7425       0.0474             0.5616                       5

Saved summary table


## Interpretation Guidelines

### What the data shows

| | Pre-LoRA | LoRA |
|---|---|---|
| Within-DB AUC | Best validation epoch (TensorBoard) | Test-time evaluation (pkl) |
| Cross-DB AUC | ❌ Not available | ✅ Full 5×5 matrix |

### How to read the delta (Δ)

- **Δ < 0** on within-DB: LoRA may slightly reduce within-DB memorization — this is *expected* because LoRA regularizes the model and prevents overfitting to a single dataset
- **Δ > 0** on within-DB: LoRA genuinely improved within-DB performance
- **Cross-DB AUC (LoRA)**: The key advantage — shows how well the model trained on one DB generalizes to completely different deepfake datasets

### Key insight

The purpose of LoRA in this project is **NOT** to improve within-DB performance (which was already high).  
The goal is to **prevent overfitting** to training-set characteristics so the model transfers better to real-world deepfakes from different sources.  

A small drop in within-DB AUC + good cross-DB AUC = LoRA is doing its job correctly.


---
## Cross-Database Evaluation

> **Pre-LoRA cross-DB results do not exist** (models were overwritten by LoRA training).  
> The matrices below show the **LoRA** 5×5 cross-DB AUC.  
> The **diagonal** cells show both: `pre-LoRA val / LoRA test` for direct comparison.


In [7]:
# =================== CELL 9: CROSS-DB MATRIX WITH PRE-LoRA DIAGONAL ===================
# Heatmap = LoRA cross-DB AUC
# Diagonal annotation = "pre-LoRA: X / LoRA: Y"

DB_KEYS = {
    'CNN-SRM':     ['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'ALL'],
    'EfficientNet':['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'ALL'],
    'ViT':         ['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'Combined'],
}
DB_LABELS = ['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'Combined/ALL']

# Pre-LoRA diagonal data from TensorBoard validation
pre_lora_diag = {
    'CNN-SRM': {
        'OpenForensics': 0.8707, 'CustomWar': 0.9985, 'CelebDF': 0.6858,
        'CiFake': None, 'ALL': None
    },
    'EfficientNet': {
        'OpenForensics': 0.9572, 'CustomWar': 0.9999, 'CelebDF': 0.7663,
        'CiFake': None, 'ALL': 0.8579
    },
    'ViT': {
        'OpenForensics': 0.5293, 'CustomWar': 0.9297, 'CelebDF': 0.7008,
        'CiFake': 0.6650, 'Combined': 0.6510
    },
}

import matplotlib.patches as mpatches, seaborn as sns, pandas as pd
import matplotlib.pyplot as plt

for model in ['CNN-SRM', 'EfficientNet', 'ViT']:
    keys = DB_KEYS[model]
    
    mat = np.full((5, 5), np.nan)
    for i, train_k in enumerate(keys):
        for j, test_k in enumerate(keys):
            r = lora_pkl[model].get(train_k, {}).get(test_k)
            if r:
                mat[i, j] = r.get('roc_auc', np.nan)
    
    fig, ax = plt.subplots(figsize=(9, 7))
    fig.suptitle(f'{model}: Cross-DB ROC-AUC (LoRA)\n'
                 f'Diagonal = pre-LoRA val / LoRA test', fontsize=12, fontweight='bold')
    
    df_mat = pd.DataFrame(mat, index=[f'Train:{d}' for d in DB_LABELS],
                          columns=[f'Test:{d}' for d in DB_LABELS])
    
    # Build custom annotation matrix
    annot = np.empty((5, 5), dtype=object)
    for i in range(5):
        for j in range(5):
            v = mat[i, j]
            if np.isnan(v):
                annot[i, j] = 'N/A'
            elif i == j:
                pre = pre_lora_diag[model].get(keys[i])
                if pre is not None:
                    annot[i, j] = f'pre:{pre:.3f}\nLoRA:{v:.3f}'
                else:
                    annot[i, j] = f'LoRA:{v:.3f}'
            else:
                annot[i, j] = f'{v:.3f}'
    
    mask_nan = np.isnan(mat)
    sns.heatmap(df_mat.astype(float), annot=annot, fmt='', cmap='Blues',
                vmin=0.3, vmax=1.0, mask=mask_nan, ax=ax,
                linewidths=0.8, cbar_kws={'label': 'ROC-AUC'},
                annot_kws={'size': 8})
    
    # Highlight diagonal
    for k in range(5):
        ax.add_patch(plt.Rectangle((k, k), 1, 1, fill=False,
                                   edgecolor='gold', lw=3, zorder=5))
    
    ax.set_xlabel('Test DB'); ax.set_ylabel('Train DB')
    ax.tick_params(axis='x', rotation=40); ax.tick_params(axis='y', rotation=0)
    
    plt.tight_layout()
    out = os.path.join(COMPARE_DIR, f'{model.lower().replace("-","_")}_cross_db_annotated.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {out}")


Saved: /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora/cnn_srm_cross_db_annotated.png


Saved: /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora/efficientnet_cross_db_annotated.png


Saved: /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora/vit_cross_db_annotated.png


In [8]:
# =================== CELL 10: CROSS-DB OFF-DIAGONAL HEATMAP ===================
# Shows only the off-diagonal (cross-DB generalization) performance for LoRA
# This is the key metric: how well does training on DB X generalize to DB Y?

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle('LoRA Cross-DB Generalization — ROC-AUC\n'
             '(Train ≠ Test • Gold border = within-DB diagonal)',
             fontsize=13, fontweight='bold')

for ax, model in zip(axes, ['CNN-SRM', 'EfficientNet', 'ViT']):
    keys = DB_KEYS[model]
    mat  = np.full((5,5), np.nan)
    for i, tk in enumerate(keys):
        for j, sk in enumerate(keys):
            r = lora_pkl[model].get(tk,{}).get(sk)
            if r: mat[i,j] = r.get('roc_auc', np.nan)
    
    df_mat = pd.DataFrame(mat, index=DB_LABELS, columns=DB_LABELS)
    
    # Color scale: 0.4 = bad, 0.7 = good, 1.0 = excellent
    mask_nan = np.isnan(mat)
    g = sns.heatmap(df_mat.astype(float), annot=True, fmt='.3f',
                cmap='RdYlGn', vmin=0.4, vmax=0.9, mask=mask_nan,
                ax=ax, linewidths=0.5, cbar_kws={'label': 'ROC-AUC'})
    
    for k in range(5):
        ax.add_patch(plt.Rectangle((k,k),1,1, fill=False, edgecolor='gold', lw=2.5, zorder=5))
    
    # Mark the best cross-DB cell
    best_i, best_j, best_v = -1, -1, -1
    for i in range(5):
        for j in range(5):
            if i != j and not np.isnan(mat[i,j]) and mat[i,j] > best_v:
                best_i, best_j, best_v = i, j, mat[i,j]
    if best_i >= 0:
        ax.add_patch(plt.Rectangle((best_j, best_i),1,1, fill=False,
                                   edgecolor='lime', lw=2.5, zorder=6, linestyle='--'))
    
    ax.set_title(model, fontsize=12, fontweight='bold')
    ax.set_xlabel('Test DB'); ax.set_ylabel('Train DB')
    ax.tick_params(axis='x', rotation=40); ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
out = os.path.join(COMPARE_DIR, 'lora_cross_db_generalization_matrix.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {out}")
print("\nGold border = within-DB, Green dashed border = best cross-DB cell")


Saved: /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/lora_vs_no_lora/lora_cross_db_generalization_matrix.png

Gold border = within-DB, Green dashed border = best cross-DB cell


In [9]:
# =================== CELL 8: SYNC TO GOOGLE DRIVE ===================
import subprocess

GDRIVE = 'gdrive:deepfake_image_project/models/lora_vs_no_lora'
r = subprocess.run(['rclone', 'sync', COMPARE_DIR, GDRIVE, '--progress', '--stats-one-line'],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
print("Sync complete →", GDRIVE)


594.370 KiB / 594.370 KiB, 100%, 0 B/s, ETA -594.370 KiB / 594.370 KiB, 100%, 0 B/s, ETA -594.370 KiB / 594.370 KiB, 100%, 594.313 KiB/s, ETA 0s

Sync complete → gdrive:deepfake_image_project/models/lora_vs_no_lora
